## Polars GPU engine

This notebook uses the Polars Lazy API with the GPU engine powered by cuDF, using the NYC Yellow Taxi dataset.

#### Installation

Choose the cudf-polars package that matches your CUDA and Python versions.

## Load the data

We load the first three months of 2023 NYC Yellow Taxi data into a Polars lazy query.

In [1]:
%load_ext cudf.pandas

import polars as pl
import polars.selectors as cs
import requests
from io import BytesIO


def load_data_polars(months=3):
    base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
    frames = []

    for month in range(1, months + 1):
        url = base_url.format(month)
        print(f"Downloading month {month:02d}...")
        response = requests.get(url)
        response.raise_for_status()

        df_month = pl.read_parquet(BytesIO(response.content))
        df_month = df_month.with_columns(cs.integer().cast(pl.Int64))
        frames.append(df_month)

    return pl.concat(frames, how="diagonal_relaxed")

df_polars = load_data_polars(months=3)
df_lazy = df_polars.lazy()

print(f"Total rows: {df_polars.height:,}")
print(f"Shape: {df_polars.shape}")

Total rows: 9,384,487
Shape: (9384487, 20)


## Enabling GPU acceleration for Polars


We'll use the same NYC taxi dataset and perform the same analysis. The only difference is that we'll express the computation as a Polars lazy query and execute it using the GPU engine.

In [2]:

query = (
    df_lazy.group_by("PULocationID").agg([
        pl.col("fare_amount").sum(),
        pl.col("trip_distance").mean(),
        pl.len().alias("trip_count"),
    ]).sort("fare_amount", descending=True)
)

display(query.collect(engine="gpu").head())

PULocationID,fare_amount,trip_distance,trip_count
i64,f64,f64,u32
132,2.7812e7,15.597877,461969
138,1.1954e7,9.690808,289100
161,6.6700e6,2.659514,431701
230,5.5109e6,3.170446,313884
237,5.4417e6,1.80937,436959


The query itself remains unchanged. The only difference is passing engine="gpu" to collect(), which tells Polars to use the GPU engine when the query is supported.

In [3]:
print("Polars CPU:")
%time summary_cpu = query.collect(engine="cpu")

print("\nPolars GPU:")
%time summary_gpu = query.collect(engine="gpu")

Polars CPU:
CPU times: user 459 ms, sys: 371 ms, total: 830 ms
Wall time: 163 ms

Polars GPU:
CPU times: user 299 ms, sys: 210 ms, total: 509 ms
Wall time: 359 ms


## Verifying GPU execution

For normal use, collect(engine="gpu") asks Polars to use the default GPU engine. When we want to be stricter, we can create a pl.GPUEngine object and pass it to collect().

Here we choose GPU device 0, which is the default in a single-GPU Colab runtime. We also set raise_on_fail=True. That makes Polars stop with an error if the query cannot run on the GPU.

In [4]:
gpu_engine = pl.GPUEngine(
    device=0,
    raise_on_fail=True,
)

summary_gpu_checked = query.collect(engine=gpu_engine)

print(
    f"GPU query completed. Input rows: {df_polars.height:,}; "
    f"grouped output rows: {len(summary_gpu_checked):,}."
)


GPU query completed. Input rows: 9,384,487; grouped output rows: 262.


## When a query can't run on the GPU

The next cell deliberately uses `map_elements`, a Python UDF the GPU engine cannot execute, to demonstrate graceful CPU fallback. The two warnings it prints are expected: `PolarsInefficientMapWarning` (Polars prefers native expressions over UDFs on any engine) and the GPU `PerformanceWarning` with `NotImplementedError: anonymousfunction` (the GPU engine reporting it fell back to the CPU). The `set_verbose(True)` is there on purpose to surface this. In real code, use a native expression instead, which does run on the GPU.

In [5]:
unsupported_gpu_operation = (
    df_lazy
    .select(
        pl.col("fare_amount")
        .map_elements(lambda fare: fare * 1.1, return_dtype=pl.Float64)
        .alias("fare_with_markup")
    )
    .head(10)
)

with pl.Config() as cfg:
    cfg.set_verbose(True)
    fallback_result = unsupported_gpu_operation.collect(engine="gpu")

print(fallback_result)

/tmp/ipykernel_22619/659847942.py:5: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("fare_amount").map_elements(lambda fare: ...)
with this one instead:
  + pl.col("fare_amount") * 1.1

  .map_elements(lambda fare: fare * 1.1, return_dtype=pl.Float64)


shape: (10, 1)
┌──────────────────┐
│ fare_with_markup │
│ ---              │
│ f64              │
╞══════════════════╡
│ 10.23            │
│ 8.69             │
│ 16.39            │
│ 13.31            │
│ 12.54            │
│ 14.08            │
│ 13.31            │
│ 50.27            │
│ 19.47            │
│ 16.39            │
└──────────────────┘


/home/avinash/codebase/python-base/gpu-programming/datascience-workflow-on-gpu-cuDF/.venv/lib/python3.14/site-packages/polars/lazyframe/frame.py:2630: PerformanceWarning: Query execution with GPU not possible: unsupported operations.
The errors were:
- NotImplementedError: anonymousfunction
  return wrap_df(ldf.collect(engine, callback))


## Scaling to multiple GPUs

If you're working with larger datasets or have access to multiple GPUs, you can switch to RayEngine with only a small change.

In [6]:
# Note: This requires 'ray' and 'cudf_polars' configured for distributed execution
# from cudf_polars.engine.ray import RayEngine
# with RayEngine() as engine:
#     summary = query.collect(engine=engine)

RuntimeError: Your /home/avinash/codebase/python-base/gpu-programming/datascience-workflow-on-gpu-cuDF/pyproject.toml is not in the working_dir /home/avinash/codebase/python-base/gpu-programming/datascience-workflow-on-gpu-cuDF/accelerating-data-preparation, so the workers will not have access to the file. Make sure the pyproject.toml file is in the working directory. You can do so by specifying --directory in 'uv run', by changing the current working directory before running 'uv run', or by using the 'working_dir' parameter of the runtime_environment.